# MERFISH GP

- Creator: Mohammad Vali Sanian (mohammad.sanian@helsinki.fi; mv10@sanger.ac.uk)
- Date of Creation: 16.08.2024
- Date of Last Modification: 21.09.2024 (by Sebastian Birk; sb75@sanger.ac.uk)

# 1. Setup

## 1.1 Load Libraries 

In [15]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
import sys
sys.path.append("../../utils")

In [17]:
import os
import logging
from datetime import datetime

import anndata as ad
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from sklearn.model_selection import train_test_split

from nichejepa.datasets import prepare_dataset
from nichejepa.infer import infer
from nichejepa.train import train
from nichejepa.utils.config import create_params_from_YAML_wandb_config
from nichejepa.utils.distributed import init_distributed
from nichejepa.utils.evaluation import clustering_metrics, classification_metrics

from analysis_utils import *

## 1.2 Load Config/Params
All setup will be done in the config file. Please edit that file to continue.

In [18]:
rank=0
logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO if rank == 0 else logging.ERROR)
params = create_params_from_YAML_wandb_config('../../../nichejepa/configs/merfish_2_gene_panels.yaml',logger)

INFO:root:Loaded parameters from YAML file.


In [19]:
params

{'data': {'batch_size': 64,
  'data_path': '/lustre/scratch126/cellgen/team361/am84/NicheJEPA/datasets/am84/st_data/gold/cell-neighborhood-tokenizer/merfish_mouse_brain_8-batch_2-gene-panels/8-shards/merfish_mouse_brain_8-batch_2-gene-panels_4_False_read_depth_shifted_log_2170.dataset',
  'split': 0.2,
  'stratify': False,
  'data_set_name': 'merfish_gp',
  'num_workers': 6,
  'pin_mem': True,
  'vocab_size': 1087,
  'seq_len_cell': 200,
  'seq_len_neighborhood': 200,
  'has_cls': False,
  'random_state': 0,
  'sample_subset': False},
 'logging': {'write_tag': 'jepa'},
 'mask': {'context_mask_size': 1220,
  'target_mask_size': 10,
  'n_contexts': 1,
  'n_targets': 9,
  'per_segment_mask_ratio': 0.2,
  'segment_masking': True},
 'meta': {'enc_depth': 3,
  'load_checkpoint': False,
  'pred_depth': 1,
  'pred_emb_dim': 768,
  'enc_emb_dim': 768,
  'enc_pred_depth': 31,
  'top_k': 127,
  'read_checkpoint': None,
  'use_bfloat16': True,
  'pos_learnable': False,
  'seg_learnable': False},
 

In [20]:
artifact_folder_path = '../../artifacts'
current_timestamp = (
    datetime.now().strftime("%d%m%Y_%H%M%S") +
    f"_{datetime.now().microsecond // 1000:03d}")
folder_path = os.path.join(artifact_folder_path,
                           params['data']['data_set_name'],
                           current_timestamp)
folder_path

'../../artifacts/merfish_gp/22092024_073752_630'

## 1.3 Create Train & Test Dataset

In [21]:
train_dataset, test_dataset = prepare_dataset(params)
train_dataset

Dataset({
    features: ['gene_tokens_cell', 'gene_tokens_neighborhood', 'cell_types', 'sample_id', 'major_brain_region', 'ccf_region_name', 'donor_id', 'batch_id', 'dataset_id', 'cell_id', 'n_nonzero_cell_tokens', 'n_nonzero_neighborhood_tokens', 'input_ids', 'n_nonzero_tokens'],
    num_rows: 199025
})

## 1.4 Run INIT Distributed

In [22]:
devices=['cuda:0']
world_size=1
os.environ['CUDA_VISIBLE_DEVICES'] = str(devices[rank].split(':')[-1])

logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO if rank == 0 else logging.ERROR)

world_size, rank = init_distributed(rank_and_world_size=(rank, world_size), port=40310)
logger.info(f'Running... (rank: {rank}/{world_size})')

INFO:root:Running... (rank: 0/1)


# 2. Model Training

In [ ]:
### Train model
train(params,
      train_dataset,
      test_dataset,
      save_folder_path=folder_path)

INFO:root:Initialized (rank/world-size) 0/1.
INFO:root:GeneTransformerEncoder(
  (gene_embed): Embedding(1087, 768, padding_idx=0)
  (seg_embed): Embedding(3, 768, padding_idx=0)
  (blocks): ModuleList(
    (0-2): 3 x Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): MLP(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (norm): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
)
INFO:roo

/lustre/scratch126/cellgen/team205/sb75/nichejepa/src/nichejepa/helper.py:290: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if use_bfloat16 else None


INFO:root:Epoch 1


/lustre/scratch126/cellgen/team205/sb75/nichejepa/src/nichejepa/train.py:409: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16,


INFO:root:[1,     0] loss: 0.526 masks: 234.0 7.0 [wd: 4.00e-02] [lr: 5.00e-05] [mem: 9.14e+03] (2074.4 ms)
INFO:root:[1,     0] grad_stats: [8.02e-03 7.20e-03] (9.30e-05, 1.23e-02)
INFO:root:[1,    10] loss: 0.452 masks: 217.6 6.5 [wd: 4.00e-02] [lr: 5.00e-05] [mem: 9.38e+03] (297.1 ms)
INFO:root:[1,    10] grad_stats: [1.03e-03 1.50e-03] (1.35e-05, 2.02e-03)
INFO:root:[1,    20] loss: 0.414 masks: 204.9 6.2 [wd: 4.00e-02] [lr: 4.99e-05] [mem: 9.38e+03] (205.8 ms)
INFO:root:[1,    20] grad_stats: [2.04e-04 4.01e-04] (2.64e-06, 6.49e-04)
INFO:root:[1,    30] loss: 0.383 masks: 200.9 6.0 [wd: 4.00e-02] [lr: 4.99e-05] [mem: 9.38e+03] (173.4 ms)
INFO:root:[1,    30] grad_stats: [9.23e-05 2.01e-04] (1.13e-06, 3.37e-04)
INFO:root:[1,    40] loss: 0.351 masks: 198.9 6.0 [wd: 4.00e-02] [lr: 4.99e-05] [mem: 9.38e+03] (157.3 ms)
INFO:root:[1,    40] grad_stats: [5.86e-05 1.30e-04] (7.96e-07, 2.19e-04)
INFO:root:[1,    50] loss: 0.321 masks: 197.1 6.0 [wd: 4.00e-02] [lr: 4.98e-05] [mem: 9.38e+03

# 3. Analysis

## 3.1 Run Model Inference

In [ ]:
#dataset = prepare_dataset(params,
#                          split_dataset=False)
#dataset

In [ ]:
# Run model inference
adata = infer(params,
              train_dataset,
              load_folder_path=folder_path,
              agg_type='avg', # 'weighted_avg', 'cls'
              cell_gene_ids=[10],
              neighborhood_gene_ids=[])

# Save adata
adata.write(f'{folder_path}/adata.h5ad')

In [ ]:
adata = sc.read_h5ad(f'{folder_path}/adata.h5ad')

## 3.2 Plot Embeddings

In [ ]:
layer = params['meta']['enc_depth'] - 1

In [ ]:
# Compute UMAP representation for (aggregated) neighborhood gene embeddings
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep=f'neighborhood_emb_layer_{layer}',
                key_added='neighborhood')
sc.tl.umap(adata,
           neighbors_key='neighborhood')

# Plot (aggregated) neighborhood gene embeddings, colored by niche
label = 'niche_types'
sc.pl.umap(adata, color=label)

# Plot (aggregated) neighborhood gene embeddings, colored by cell type
label = 'cell_types'
sc.pl.umap(adata, color=label)

# Plot (aggregated) neighborhood gene embeddings, colored by batch
label = 'batch_id'
sc.pl.umap(adata, color=label)

In [ ]:
# Compute UMAP representation for (aggregated) cell gene embeddings
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep=f'cell_emb_layer_{layer}',
                key_added='cell')
sc.tl.umap(adata,
           neighbors_key='cell')

# Plot (aggregated) cell gene embeddings, colored by niche
label = 'niche_types'
sc.pl.umap(adata, color=label)

# Plot (aggregated) cell gene embeddings, colored by cell type
label = 'cell_types'
sc.pl.umap(adata, color=label)

# Plot (aggregated) cell gene embeddings, colored by batch
label = 'batch_id'
sc.pl.umap(adata, color=label)

## 3.3 Plot Embedding Similarity Stats

In [ ]:
similarity_stats = compute_similarity_stats(
    adata,
    n_samples=None,
    type_key='cell_type',
    embedding_key=f'cell_emb_layer_{layer}')
plot_similarity_heatmaps(similarity_stats)

## 3.4 Compute Metrics

In [ ]:
# NMI & ARI for cell types (cell embedding)
clustering_metrics(adata,
                   emb_key=f'cell_emb_layer_{layer}',
                   label_col='cell_type')

In [ ]:
# NMI & ARI for niche (neighborhood embedding)
clustering_metrics(adata,
                   emb_key=f'neighborhood_emb_layer_{layer}',
                   label_col='niche')

## 3.5 Plot Loss Curve

In [ ]:
df = pd.read_csv(f'{folder_path}/jepa_r0.csv')
df['loss'] = pd.to_numeric(df['loss'], errors='coerce')

plt.figure(figsize=(10, 6))
plt.plot(df.index,df['loss'])
plt.title('Loss Function Over Iterations')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.grid(True)
plt.show()